In [ ]:
!pip show openai

In [ ]:
import sys
from os.path import dirname
sys.path.append('/home/farlab/miniconda3/lib/python3.10/site-packages')

In [ ]:
#imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#import seaborn as sns
import os
import json
import openai
#from scipy.interpolate import interp1d
#import mse
#from sklearn.metrics import mean_squared_error
#from scipy.signal import medfilt 
import time

In [ ]:
#make new folder, called "transcript_dfs"
os.system('mkdir transcript_dfs')

In [ ]:
#go to folder

path = './transcripts/'


listfiles = os.listdir(path)

for file in listfiles:
    #open vtt file
    with open(path+file) as f:
        lines = f.readlines()
    #delete first line
    lines = lines[1:]
    #delete lines that are empty ('\n)
    lines = [x for x in lines if x != '\n']

    #start dataframe
    df = pd.DataFrame(columns=['start','end','speaker','text'])

    #loop through lines
    for line in lines:
        #remove '\n' from end of line
        if line != lines[-1]:
            line = line[:-1]
        #if line starts with a number, it is a timestamp
        if line[0].isdigit():
            #split line into start and end time
            start = line.split(' --> ')[0]
            end = line.split(' --> ')[1]
            #print(start,end)
        #if line starts with a letter, it is a speaker
        elif line[0].isalpha():
            #split line into speaker and text
            speaker = line.split(':')[0]
            text = line.split(':')[1]
            #print(speaker,text)
            #add row to dataframe
            df = df.append({'start':start,'end':end,'speaker':speaker,'text':text},ignore_index=True)

    #save dataframe as csv
    df.to_csv('./transcript_dfs/'+file[:-8]+'.csv',index=False)

    print(df.head())
    
    

In [ ]:
df

In [ ]:
import re

def get_sentiment(text):

  prompt_text = """Classify the sentiment of the following earnings calls text as as positive, negative, or neutral. 
  text: {}
  sentiment: """.format(text)

  sentiment = openai.Completion.create(
              model="text-davinci-003",
              prompt = prompt_text,
              max_tokens= 15,
              temperature=0,
              )
  
  # remove special characters e.g n etc, from response
  sentiment = re.sub('W+','', sentiment['choices'][0]['text'])
  return sentiment

In [ ]:
#temporary
openai.api_key = 'REDACTED_OPENAI_API_KEY'

In [ ]:
#merge all dfs into one
path = './transcript_dfs/'
files = os.listdir(path)
all_df = pd.DataFrame()
for file in files:
    print(file)
    if 'all' not in file:
        #open csv file
        df = pd.read_csv(path+file)
        df['session'] = [file[:-4]]*len(df)
        all_df = pd.concat([all_df,df], axis = 0)
        print(all_df.size)
#save as new csv file
all_df.to_csv(path+'all_df.csv',index=False)

In [ ]:
#DO NOT USE
#open each csv file and run function "get_sentiment" on each line. Then, add sentiment to the dataframe and save as new csv file
path = './transcript_dfs/'
files = os.listdir(path)
files = files[2:]
for file in files:
    print(file)
    #open csv file
    df = pd.read_csv(path+file)
    #get sentiment
    df['sentiment'] = df['text'].apply(get_sentiment)
    print(df)
    #save as new csv file
    df.to_csv(path+file,index=False)

In [ ]:
#open each csv file and run function "get_sentiment" on each line. Then, add sentiment to the dataframe and save as new csv file
path = './transcript_dfs/'
file = 'all_df.csv'
df = pd.read_csv(path+file)

#new df, copy of df but add sentiment column
df_sentiment = pd.read_csv(path+'all_df_sentiment.csv')
#split into 100 lines at a time
linelist = np.arange(300,len(df),60)
print(linelist)
for i in range(len(linelist)):
    if i == len(linelist)-1:
        df_line = df.iloc[linelist[i]:len(df)-1]

        #get sentiment
        df_line['sentiment'] = df_line['text'].apply(get_sentiment)
        #save in new df
        df_sentiment = pd.concat([df_sentiment,df_line], axis = 0)
        print(i)
        #save as new csv file
        df_sentiment.to_csv(path+'all_df_sentiment.csv',index=False)
        print(df_sentiment)
        #wait 30 seconds
        time.sleep(45)
    elif i>70:
        df_line = df.iloc[linelist[i]:linelist[i+1]]

        #get sentiment
        df_line['sentiment'] = df_line['text'].apply(get_sentiment)
        #save in new df
        df_sentiment = pd.concat([df_sentiment,df_line], axis = 0)
        print(i)
        #save as new csv file
        df_sentiment.to_csv(path+'all_df_sentiment.csv',index=False)
        print(df_sentiment)
        #wait 30 seconds
        time.sleep(45)
    

In [ ]:
len(df)